In [1]:
import sys
sys.path.append('../')

import numpy as np
from utils_m2_factorize import (
    expand_tensor_product, 
    expand_tensor_product_for_incomplete_qubit_set,
    partition_from_dict,
    obtain_join_partition,
    obtain_coarse_dicts,
    QC_assignment_from_qubit_labels
)
from numpy.random import uniform

Test `partition_from_dict`

In [2]:
factorization_dict = {
    (1,2,3)   : uniform(-1, 1, 2**3),
    (4,)      : uniform(-1, 1, 2**1),
    (5,6,7,8) : uniform(-1, 1, 2**4)
}

print(partition_from_dict(factorization_dict))

factorization_dict = {
    (1,20,22)     : uniform(-1, 1, 2**3),
    (12,14,17,19) : uniform(-1, 1, 2**4),
    (3,5,7,23)    : uniform(-1, 1, 2**4),
    (13,)         : uniform(-1, 1, 2**1)
}

print(partition_from_dict(factorization_dict))

[(1, 2, 3), (4,), (5, 6, 7, 8)]
[(1, 20, 22), (12, 14, 17, 19), (3, 5, 7, 23), (13,)]


Test `obtain_join_partition`

In [3]:
p1 = [(0,), (1,2,3,4,5,6), (7,8)]
p2 = [(0,1), (2,3,4,5,6), (7,), (8,)]

print(obtain_join_partition(p1, p2), '\n')

p1 = [(0,1,2), (3,4,5,6), (7,), (8,), (9,10), (11,12)]
p2 = [(0,1), (2,3,4,5), (6,7), (8,), (9,10,11), (12,)]

print(obtain_join_partition(p1, p2), '\n')

p1 = [(0,4,7,9), (1,10,12,15), (2,3,6), (8,14), (11,), (13,), (5,)]
p2 = [(0,5,10,15), (1,3,7), (2,), (4,), (6,), (8,), (9,11,12), (13,), (14,)]

print(obtain_join_partition(p1, p2))

[(0, 1, 2, 3, 4, 5, 6), (7, 8)] 

[(0, 1, 2, 3, 4, 5, 6, 7), (8,), (9, 10, 11, 12)] 

[(0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 15), (8, 14), (13,)]


Test `obtain_coarse_dicts`. The test is as follows

1. Create two `factorization_dicts`
2. Apply `obtain_coarse_dicts` to them.
3. Check if the keys correspond to the join of the original partitions.
4. Check if the coarse_dicts encode the same state as the original dicts.

In [4]:
fd1 = {
    (0,)          : uniform(-1, 1, 2**1),
    (1,2,3,4,5,6) : uniform(-1, 1, 2**6),
    (7,8)         : uniform(-1, 1, 2**2)
}

fd2 = {
    (0,1)       : uniform(-1, 1, 2**2),
    (2,3,4,5,6) : uniform(-1, 1, 2**5),
    (7,)        : uniform(-1, 1, 2**1),
    (8,)        : uniform(-1, 1, 2**1)
}

join, cd1, cd2 = obtain_coarse_dicts(fd1, fd2)

assert join == obtain_join_partition(partition_from_dict(fd1), partition_from_dict(fd2))
assert np.allclose(expand_tensor_product(fd1, 9), expand_tensor_product(cd1, 9))
assert np.allclose(expand_tensor_product(fd2, 9), expand_tensor_product(cd2, 9))
assert np.allclose(expand_tensor_product_for_incomplete_qubit_set(fd1), expand_tensor_product_for_incomplete_qubit_set(cd1))
assert np.allclose(expand_tensor_product_for_incomplete_qubit_set(fd2), expand_tensor_product_for_incomplete_qubit_set(cd2))


fd1 = {
    (0,1,2)   : uniform(-1, 1, 2**3),
    (3,4,5,6) : uniform(-1, 1, 2**4),
    (7,)      : uniform(-1, 1, 2**1),
    (8,)      : uniform(-1, 1, 2**1),
    (9,10)    : uniform(-1, 1, 2**2),
    (11,12)   : uniform(-1, 1, 2**2)
}

fd2 = {
    (0,1)     : uniform(-1, 1, 2**2),
    (2,3,4,5) : uniform(-1, 1, 2**4),
    (6,7)     : uniform(-1, 1, 2**2),
    (8,)      : uniform(-1, 1, 2**1),
    (9,10,11) : uniform(-1, 1, 2**3),
    (12,)     : uniform(-1, 1, 2**1)
}

join, cd1, cd2 = obtain_coarse_dicts(fd1, fd2)

assert join == obtain_join_partition(partition_from_dict(fd1), partition_from_dict(fd2))
assert np.allclose(expand_tensor_product(fd1, 13), expand_tensor_product(cd1, 13))
assert np.allclose(expand_tensor_product(fd2, 13), expand_tensor_product(cd2, 13))
assert np.allclose(expand_tensor_product_for_incomplete_qubit_set(fd1), expand_tensor_product_for_incomplete_qubit_set(cd1))
assert np.allclose(expand_tensor_product_for_incomplete_qubit_set(fd2), expand_tensor_product_for_incomplete_qubit_set(cd2))

Verify `QC_assignment_from_qubit_labels`

In [5]:
p1 = [(0,), (1,2,3,4,5,6), (7,8)]
l1 = {
    0 : 'N',
    1 : 'W',
    2 : 'W',
    3 : 'W',
    4 : 'W',
    5 : 'W',
    6 : 'W',
    7 : 'V',
    8 : 'V'
}
p2 = [(0,1), (2,3,4,5,6), (7,), (8,)]
l2 = {
    0 : 'N',
    1 : 'N',
    2 : 'W',
    3 : 'W',
    4 : 'W',
    5 : 'W',
    6 : 'W',
    7 : 'N',
    8 : 'V'
}

join          = obtain_join_partition(p1, p2)
QC_assignment = QC_assignment_from_qubit_labels(l1, l2, join)

print(join)
print(QC_assignment)
print()


p1 = [(0,1,2), (3,4,5,6), (7,), (8,), (9,10), (11,12)]
l1 = {
    0  : 'W',
    1  : 'W',
    2  : 'W',
    3  : 'V',
    4  : 'V',
    5  : 'V',
    6  : 'V',
    7  : 'N',
    8  : 'N',
    9  : 'V',
    10 : 'V',
    11 : 'N',
    12 : 'N'
}

p2 = [(0,1), (2,3,4,5), (6,7), (8,), (9,10,11), (12,)]
l2 = {
    0  : 'W',
    1  : 'W',
    2  : 'V',
    3  : 'V',
    4  : 'V',
    5  : 'V',
    6  : 'W',
    7  : 'W',
    8  : 'N',
    9  : 'V',
    10 : 'V',
    11 : 'V',
    12 : 'W'
}

join          = obtain_join_partition(p1, p2)
QC_assignment = QC_assignment_from_qubit_labels(l1, l2, join)

print(join)
print(QC_assignment)
print()


[(0, 1, 2, 3, 4, 5, 6), (7, 8)]
{(0, 1, 2, 3, 4, 5, 6): 'Q', (7, 8): 'C'}

[(0, 1, 2, 3, 4, 5, 6, 7), (8,), (9, 10, 11, 12)]
{(0, 1, 2, 3, 4, 5, 6, 7): 'Q', (8,): 'C', (9, 10, 11, 12): 'Q'}

